In [1]:
import logging

import pandas as pd

import data.helpers as dh
import models.var_builders as var_builders
import data.cfr_data_19_23 as cfrd
import data.breathe_data as bd

Exploring the CF Trust registry to evaluate if model output (synthesizing FEV1, FEF25-75 on 2 days) can be used to improve ML achieved from each indidivually 

# Load, process, save yearly data

In [2]:
# Load 2019 data
df19 = cfrd.build_cfr_df(2019)

INFO:root:Loaded {df.shape[0]} entries
INFO:root:2046 entries after removing <18yr


In [4]:
df19.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_19_processed.xlsx",
    index=False,
)

In [2]:
df23 = cfrd.build_cfr_df(2023)

INFO:root:Loaded 10344 entries
INFO:root:5065 after removing all NaN
INFO:root:2701 entries after removing <18yr


In [7]:
df23.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_23_processed.xlsx",
    index=False,
)

# Link 2019 with 2023 data

In [12]:
df19 = bd.load_meas_from_excel("CF_Registry_19_processed", study_folder='CFR')

INFO:root:* Checking for same day measurements *


In [9]:
df23

,ID,Age,Height,FEV1,FEF2575,Sex,Date Recorded,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,ecFEV1 % Predicted,FEV1 % Predicted
0,B155916,36,164.0,1.90,0.75,Female,2023-01-01,1.90,0.75,39.473685,3.166484,60.003459,60.003459
1,B155917,50,174.0,2.76,1.18,Female,2023-01-01,2.76,1.18,42.753621,3.195502,86.371404,86.371404
2,B155918,38,193.0,4.78,3.35,Female,2023-01-01,4.78,3.35,70.083677,4.409788,108.395228,108.395228
3,B155920,42,171.0,4.51,4.08,Female,2023-01-01,4.51,4.08,90.465626,3.304481,136.481353,136.481353
4,B155921,38,150.0,1.37,0.59,Female,2023-01-01,1.37,0.59,43.065691,2.583624,53.026292,53.026292
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10295,C225784,22,185.0,4.90,6.06,Female,2023-01-01,4.90,6.06,123.673466,4.366308,112.222952,112.222952
10300,C225840,25,169.0,3.06,2.74,Female,2023-01-01,3.06,2.74,89.542486,3.563364,85.873907,85.873907
10304,C225865,56,174.0,1.80,0.73,Female,2023-01-01,1.80,0.73,40.555558,3.013003,59.741070,59.741070
10317,C225974,32,180.0,3.24,2.48,Female,2023-01-01,3.24,2.48,76.543210,3.950008,82.025146,82.025146


In [17]:
df23

,ID,Age,Height,FEV1,FEF2575,Sex,Date Recorded,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,ecFEV1 % Predicted,FEV1 % Predicted
0,B155916,36,164.0,1.90,0.75,Female,2023-01-01,1.90,0.75,39.473685,3.166484,60.003459,60.003459
1,B155917,50,174.0,2.76,1.18,Female,2023-01-01,2.76,1.18,42.753621,3.195502,86.371404,86.371404
2,B155918,38,193.0,4.78,3.35,Female,2023-01-01,4.78,3.35,70.083677,4.409788,108.395228,108.395228
3,B155920,42,171.0,4.51,4.08,Female,2023-01-01,4.51,4.08,90.465626,3.304481,136.481353,136.481353
4,B155921,38,150.0,1.37,0.59,Female,2023-01-01,1.37,0.59,43.065691,2.583624,53.026292,53.026292
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10295,C225784,22,185.0,4.90,6.06,Female,2023-01-01,4.90,6.06,123.673466,4.366308,112.222952,112.222952
10300,C225840,25,169.0,3.06,2.74,Female,2023-01-01,3.06,2.74,89.542486,3.563364,85.873907,85.873907
10304,C225865,56,174.0,1.80,0.73,Female,2023-01-01,1.80,0.73,40.555558,3.013003,59.741070,59.741070
10317,C225974,32,180.0,3.24,2.48,Female,2023-01-01,3.24,2.48,76.543210,3.950008,82.025146,82.025146


In [24]:
df = pd.concat([df19, df23]).sort_values('ID')

In [43]:
(df.groupby('ID').apply(lambda df: len(df)) > 1).sum()

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_32318/3045146862.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  (df.groupby('ID').apply(lambda df: len(df)) > 1).sum()


1485

In [78]:
df = bd.load_meas_from_excel("CF_Registry_19_23_processed_with_idx", study_folder="CFR")

INFO:root:* Checking for same day measurements *


In [81]:
ids_with_2_entries = df['ID'].value_counts()[df['ID'].value_counts() == 2].index.tolist()
df = df[df.ID.isin(ids_with_2_entries)]

df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_19_23_stricly_2_entries_processed_with_idx.xlsx",
    index=False,
)

# Add obs indices

In [25]:
# Add indices for model
height = df.Height.iloc[0]
age = df.Age.iloc[0]
sex = df.Sex.iloc[0]
ar_prior = "uniform"
ecfev1_noise_model_cpt_suffix = "_std_add_mult_ecfev1"
ar_fef2575_cpt_suffix = "_ecfev1_2_days_model_add_mult_noise"
(
    HFEV1,
    uFEV1,
    ecFEV1,
    AR,
    ecFEF2575prctecFEV1,
) = var_builders.fev1_fef2575_point_in_time_model_noise_shared_healthy_vars(
    height,
    age,
    sex,
    ar_prior,
    ecfev1_noise_model_cpt_suffix,
    ar_fef2575_cpt_suffix,
)

df[f"idx {ecFEV1.name}"] = df.apply(
    lambda row: ecFEV1.get_bin_idx_for_value(row["ecFEV1"]), axis=1
)
df[f"idx {ecFEF2575prctecFEV1.name}"] = df.apply(
    lambda row: ecFEF2575prctecFEV1.get_bin_idx_for_value(row["ecFEF2575%ecFEV1"]),
    axis=1,
)

In [26]:
df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_19_23_processed_with_idx.xlsx",
    index=False,
)

## Load CF complications variables

In [ ]:
cols2read = ["s01caseid_original", "s02hospivqty", "s02homeivqty"]
df = cfrd.load_cfr_data('2019', cols2read)

In [11]:
colnames = ['ID', 'Home IVs', 'Hosp IVs']
df = df.rename(columns = dict(zip(cols2read, colnames)))

In [16]:
df.describe()

,Home IVs,Hosp IVs
count,10063.000000,10063.000000
mean,0.805525,0.488920
std,1.432799,1.130894
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,1.000000,0.000000
max,20.000000,11.000000
